In [ ]:
import neat
import numpy as np

from scipy.integrate import solve_ivp

In [ ]:
G_MAX = 1
U_MAX = 5.0
X_INIT = np.array([0.0,   0.0, 0.0, 0.0])
X_GOAL = np.array([np.pi, 0.0, 0.0, 0.0])
SECONDS = 10
HZ = 50

In [ ]:
GRAVITATIONAL_ACCELERATION = 9.81

ACROBOT_SUTTON_PARAMS = {
    'm1': 1.0, 'l1': 1.0, 'lc1': 0.5, 'i1': 1.0,
    'm2': 1.0, 'l2': 1.0, 'lc2': 0.5, 'i2': 1.0,
    'g': GRAVITATIONAL_ACCELERATION
}

In [ ]:
class Acrobot():

    def __init__(self, dt=0.05):
        self.state = np.array([0.0, 0.0, 0.0, 0.0])
        self.params = ACROBOT_SUTTON_PARAMS
        self.dt = dt

    def reset(self):
        self.state = np.array([0.0, 0.0, 0.0, 0.0])
        return self.state.copy()

    def step(self, torque):
        new_state = Acrobot.dynamics(self.state, torque, self.params)
        self.state = self.state + new_state * self.dt
        return self.state.copy()

    @staticmethod
    def dynamics(state, torque, params):
        theta1, theta2, theta1_dot, theta2_dot = state
        m1, l1, lc1, i1 = params['m1'], params['l1'], params['lc1'], params['i1']
        m2, l2, lc2, i2 = params['m2'], params['l2'], params['lc2'], params['i2']
        g = params['g']

        s2 = np.sin(theta2)
        c2 = np.cos(theta2)

        phi2 = m2 * lc2 * g * np.cos(theta1 + theta2 - np.pi/2);
        phi1 = - m2 * l1 * lc2 * theta2_dot**2 * s2 \
                - 2 * m2 * l1 * lc2 * theta2_dot * theta1_dot * s2 \
                + (m1 * lc1 + m2 * l1) * g * np.cos(theta1 - np.pi/2) + phi2;
        d2 = m2 * (lc2**2 + l1 * lc2 * c2) + i2;
        d1 = m1 * lc1**2 + m2 * (l1**2 + lc2**2 + 2 * l1 * lc2 * c2) + i1 + i2;

        theta2_ddot = (torque + d2 / d1 * phi1 - phi2) / (m2 * lc2**2 + i2 - d2**2 / d1);
        theta1_ddot = - (d2 * theta2_ddot + phi1) / d1;

        return np.array([theta1_dot, theta2_dot, theta1_ddot, theta2_ddot])

In [ ]:
def compute_torque(x_i, net, u_max):
    theta1_dot = np.clip(x_i[2], -20, 20) / 20
    theta2_dot = np.clip(x_i[3], -20, 20) / 20
    x_input = np.array([np.sin(x_i[0]), np.cos(x_i[0]), np.sin(x_i[1]), np.cos(x_i[1]), theta1_dot, theta2_dot])
    u_val_net = u_max * net.activate(x_input)[0]
    return u_val_net


def acrobot_ode(t_current, x_in, net, u_max):
    u_val_net = compute_torque(x_in, net, u_max)
    x_dot = Acrobot.dynamics(x_in, u_val_net, ACROBOT_SUTTON_PARAMS)
    return x_dot


def run_experiment(net, x_init, u_max):
    t_span = (0, SECONDS)
    t_eval = np.linspace(t_span[0], t_span[1], HZ * t_span[1])
    sol = solve_ivp(
        acrobot_ode,
        t_span,
        x_init,
        method='RK45',
        t_eval=t_eval,
        args=(net, u_max),
        rtol=1e-6,
        atol=1e-8
    )
    return sol

In [ ]:
def fitness_function(state, steps):
    theta1, theta2, theta1_dot, theta2_dot = state
    return 0.0

In [ ]:
def eval_net(net, x_init, x_goal, u_max):
    fitness = 0.0
    sol = run_experiment(net, x_init, u_max)
    if sol.success:
        for i in range(sol.y.shape[1]):
            fitness += fitness_function(sol.y[:,i], i)
    return fitness


def eval_genomes(genomes, config):
    for genome_id, genome in genomes:
        net = neat.nn.FeedForwardNetwork.create(genome, config)
        genome.fitness = eval_net(net, X_INIT, X_GOAL, U_MAX)


def compute_sol_torque(sol, net, x_goal, u_max):
    torques = []
    for i in range(sol.y.shape[1]):
        x_i = sol.y[:, i]
        u_val_net = compute_torque(sol.y[:, i], net, u_max)
        torques.append(u_val_net)
    return torques

In [ ]:
import matplotlib
import matplotlib.animation as animation
import matplotlib.pyplot as plt


def plot_acrobot_state(sol, torques=None):
    if torques:
        plt.figure(figsize=(12, 8))
        plt.subplot(2, 1, 1)
    else:
        plt.figure(figsize=(12, 4))

    plt.plot(sol.t, sol.y[0, :], label=r'$\theta_1$')
    plt.plot(sol.t, sol.y[1, :], label=r'$\theta_2$')
    plt.plot(sol.t, sol.y[2, :], color='C0', linestyle=(0,(5,5)), label=r'$\dot{\theta}_1$')
    plt.plot(sol.t, sol.y[3, :], color='C1', linestyle=(0,(5,5)), label=r'$\dot{\theta}_2$')
    plt.axhline(y=np.pi, color='#808080', linestyle='--')
    plt.axhline(y=0, color='#808080', linestyle='--')
    plt.title('Acrobot State vs. Time')
    plt.xlabel('time (s)')
    plt.ylabel('state (rad, rad/s)')
    plt.legend()
    plt.grid(True)

    if torques:
        plt.subplot(2, 1, 2)
        plt.plot(sol.t, torques, label=r'$u$')
        plt.title('Control Torque vs. Time')
        plt.xlabel('time (s)')
        plt.ylabel('torque (Nm)')
        plt.legend()
        plt.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
config_file = 'neat-ff.cfg'

config = neat.Config(
    neat.DefaultGenome,
    neat.DefaultReproduction,
    neat.DefaultSpeciesSet,
    neat.DefaultStagnation,
    config_file)
p = neat.Population(config)

p.add_reporter(neat.StdOutReporter(False))

winner = p.run(eval_genomes, G_MAX)

print('\nBest genome:\n{!s}\n'.format(winner))
winner_net = neat.nn.FeedForwardNetwork.create(winner, config)
sol = run_experiment(winner_net, X_INIT, U_MAX)
if sol.success:
    torque = compute_sol_torque(sol, winner_net, X_GOAL, U_MAX)
    plot_acrobot_state(sol, torque)